In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow

In [15]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

In [16]:
customer_orders = orders.merge(customers, on= 'customer_id', how = 'left')
customer_orders_value = customer_orders.merge(order_payments, on = 'order_id', how = 'left')
customer_orders_value_review = customer_orders_value.merge(order_reviews, on = 'order_id', how = 'left')

In [17]:
customer_orders_value_review.sample(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
86308,17dd29c2a9196baed66fc07147b4d819,d291d348d3df231748e584445f34fd98,delivered,2018-02-26 14:03:27,2018-02-27 14:09:00,2018-02-28 23:37:16,2018-03-29 19:29:07,2018-03-16 00:00:00,03a5b2c623827a7d4e727ac0aa2284d4,21610,...,1.0,credit_card,3.0,200.55,3d2eb84738ec67cee204e322b2294ab5,1.0,NaN,"Quero cancelar esta compra, pois já passou do ...",2018-03-21 00:00:00,2018-03-21 14:35:43
77001,056bfadd41b8600ad5ecfef2ac132188,91faeb032f47277f654a8202eaa9fed6,delivered,2017-11-25 11:06:38,2017-11-25 11:15:31,2017-11-27 19:56:58,2017-12-04 14:44:45,2017-12-15 00:00:00,2106e7aa97e2b0bb0193c9f2478d85e0,9895,...,1.0,credit_card,3.0,35.01,acf3f5b4d93975335ee6a76654cd495b,4.0,NaN,NaN,2017-12-05 00:00:00,2017-12-09 13:53:28
57180,de4c5454fb8e64f76ec9689a472b23c7,66b1555c5aaf32eda9799f8c6af3a884,delivered,2017-11-24 14:45:59,2017-11-24 16:52:42,2017-11-27 16:12:05,2017-12-13 15:25:20,2017-12-19 00:00:00,9d85f9ce22312bcc36d6d8634ecc1e9b,13370,...,1.0,credit_card,4.0,235.88,eb85781c9ce7cb0ca776f80d5c1b0118,3.0,NaN,"NO DIA 24/11 EFETUEI UMA COMPRA DE 2 BONECAS, ...",2017-12-14 00:00:00,2017-12-15 17:18:11
46177,232f30270c13607c2184ab9980245bfc,bc7eb769eeb68a51edf0205462dc15de,delivered,2018-08-01 16:44:36,2018-08-03 04:35:18,2018-08-04 10:51:00,2018-08-08 21:51:55,2018-08-15 00:00:00,db5926d4ade42690095dd9e633ca8e5b,89082,...,1.0,boleto,1.0,94.72,ecb10e3f87dd26c443d34383df1d2a9d,5.0,NaN,NaN,2018-08-09 00:00:00,2018-08-10 10:13:43
77982,f85b00b88275bb213518b0ebacca0544,d7c70fe22a435b47ba0bab4bb6ef7883,delivered,2017-09-15 11:26:04,2017-09-15 11:44:16,2017-09-22 18:18:15,2017-09-29 19:22:44,2017-10-06 00:00:00,59b90826e4ef0c0001e9d17359bf219e,37470,...,1.0,credit_card,1.0,67.60,36f293d57e410d58088f3dfe38da5ead,5.0,NaN,NaN,2017-09-30 00:00:00,2017-10-01 15:47:49


In [18]:
print(f"{customer_orders_value_review['payment_value'].sum()} is the total amount of revenue")

16081420.74 is the total amount of revenue


In [ ]:
# Make payments one row per order
payments_agg = order_payments.groupby('order_id', as_index=False)['payment_value'].sum()

# Make reviews one row per order
reviews_agg = order_reviews.groupby('order_id', as_index=False)['review_score'].mean()

# Merge into ONE row per order
customer_orders_value_review = (
    orders
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_agg, on='order_id', how='left')
)

In [ ]:
customer_orders_value_review['Count'] = customer_orders_value_review.groupby('customer_unique_id')['customer_unique_id'].transform('count')
repeat_customers = customer_orders_value_review[customer_orders_value_review['Count'] > 1]
repeat_customers_save = repeat_customers.copy()
repeat_customers = repeat_customers[repeat_customers['order_status'] == 'delivered']


In [21]:
repeat_customers.sample(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,payment_value,review_score,Count
87175,ba9ec6bfbbe8454a533f194e8cd420f8,f3201a8a4ea32786b7b5948da7e7a50b,delivered,2018-03-28 16:09:21,2018-03-28 16:27:50,2018-03-29 23:54:35,2018-04-04 17:51:04,2018-04-17 00:00:00,935b9c5a3162185f88dac06d8d08d623,136.30,5.0,3
43093,5eee483d18d84faaef4bea7e8b4579df,a719472fec09c5540edb0bae9cf81e44,delivered,2018-03-15 15:21:30,2018-03-15 15:35:29,2018-03-16 22:11:12,2018-03-26 19:33:01,2018-04-17 00:00:00,4a433c62dfda8104d295307e855c82a7,164.36,5.0,2
84934,fae7ec9e7beff1d7e8ba7161738e557f,a95a14821505950abf17a562b31d1bf9,delivered,2017-06-19 20:18:51,2017-06-21 02:15:35,2017-06-23 08:35:48,2017-06-28 13:22:06,2017-07-11 00:00:00,1a15eecd72a08e1134ba406c7a7998e6,131.75,3.0,2


Proper Datatypes and Sorting

In [22]:
df = repeat_customers.copy()
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])
df = df.sort_values(['customer_unique_id', 'order_purchase_timestamp'])

In [ ]:
import pandas as pd
import numpy as np


def clean_delivery_delay_data(
    df: pd.DataFrame,
    purchase_col: str = 'order_purchase_timestamp',
    delivered_col: str = 'order_delivered_customer_date',
    estimated_col: str = 'order_estimated_delivery_date',
    delay_col: str = 'delivery_delay_days',
    method: str = 'iqr',
    iqr_multiplier: float = 1.5,
    lower_quantile: float = 0.01,
    upper_quantile: float = 0.99,
    drop_invalid: bool = True,
    clip_valid_delays: bool = True,
    copy: bool = True
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Clean delivery delay data using:
    1) logical consistency checks
    2) winsorization / clipping of valid rows

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    purchase_col, delivered_col, estimated_col, delay_col : str
        Column names.
    method : str
        'iqr' for IQR-based clipping or 'quantile' for quantile clipping.
    iqr_multiplier : float
        Multiplier for IQR clipping when method='iqr'.
    lower_quantile, upper_quantile : float
        Quantiles for clipping when method='quantile'.
    drop_invalid : bool
        If True, remove logically invalid rows.
    clip_valid_delays : bool
        If True, clip delay_col after logical filtering.
    copy : bool
        If True, work on a copy.

    Returns
    -------
    cleaned_df : pd.DataFrame
        Cleaned dataframe.
    invalid_rows : pd.DataFrame
        Rows flagged as logically invalid.
    summary : dict
        Summary of checks and clipping bounds.
    """

    if copy:
        df = df.copy()

    required_cols = [purchase_col, delivered_col, estimated_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    for col in [purchase_col, delivered_col, estimated_col]:
        df[col] = pd.to_datetime(df[col], errors='coerce')

  
    df['expected_delivery_days'] = (df[estimated_col] - df[purchase_col]).dt.days


    if delay_col not in df.columns:
        df[delay_col] = (df[delivered_col] - df[estimated_col]).dt.days
    else:
        df[delay_col] = pd.to_numeric(df[delay_col], errors='coerce')

   
    df['flag_missing_core_dates'] = (
        df[purchase_col].isna() |
        df[delivered_col].isna() |
        df[estimated_col].isna()
    )

    df['flag_estimated_before_purchase'] = (
        df[estimated_col] < df[purchase_col]
    )

    df['flag_delivered_before_purchase'] = (
        df[delivered_col] < df[purchase_col]
    )


    df['flag_earlier_than_feasible_window'] = (
        df['expected_delivery_days'].notna() &
        df[delay_col].notna() &
        (df[delay_col] < -df['expected_delivery_days'])
    )

    df['is_invalid_delivery_record'] = (
        df['flag_missing_core_dates'] |
        df['flag_estimated_before_purchase'] |
        df['flag_delivered_before_purchase'] |
        df['flag_earlier_than_feasible_window']
    )

    invalid_rows = df[df['is_invalid_delivery_record']].copy()

    if drop_invalid:
        valid_df = df[~df['is_invalid_delivery_record']].copy()
    else:
        valid_df = df.copy()

    clipping_info = {
        'method': method,
        'lower_bound': None,
        'upper_bound': None
    }

    if clip_valid_delays:
        valid_delay = valid_df[delay_col].dropna()

        if len(valid_delay) == 0:
            raise ValueError("No valid delivery delay values available for clipping.")

        if method == 'iqr':
            q1 = valid_delay.quantile(0.25)
            q3 = valid_delay.quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - iqr_multiplier * iqr
            upper_bound = q3 + iqr_multiplier * iqr

        elif method == 'quantile':
            lower_bound = valid_delay.quantile(lower_quantile)
            upper_bound = valid_delay.quantile(upper_quantile)

        else:
            raise ValueError("method must be either 'iqr' or 'quantile'")

        valid_df[f'{delay_col}_raw'] = valid_df[delay_col]
        valid_df[delay_col] = valid_df[delay_col].clip(lower=lower_bound, upper=upper_bound)

        clipping_info['lower_bound'] = float(lower_bound)
        clipping_info['upper_bound'] = float(upper_bound)

    summary = {
        'n_input_rows': int(len(df)),
        'n_invalid_rows': int(len(invalid_rows)),
        'pct_invalid_rows': float(len(invalid_rows) / len(df)) if len(df) > 0 else np.nan,
        'n_output_rows': int(len(valid_df)),
        'n_missing_core_dates': int(df['flag_missing_core_dates'].sum()),
        'n_estimated_before_purchase': int(df['flag_estimated_before_purchase'].sum()),
        'n_delivered_before_purchase': int(df['flag_delivered_before_purchase'].sum()),
        'n_earlier_than_feasible_window': int(df['flag_earlier_than_feasible_window'].sum()),
        'clipping': clipping_info
    }

    return valid_df, invalid_rows, summary

In [24]:
df = clean_delivery_delay_data(df)[0]

In [25]:
base_df = df.copy()

base_df = base_df[
    [
        'order_id',
        'customer_unique_id',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'review_score',
        'payment_value'
    ]
].copy()

base_df.to_parquet('../data/processed/base_orders.parquet', index=False)